# 30 · Scale-Up Experiments (Reviewer Fixes #7–9)

Addresses the three "needs modest additional work" items from review feedback:

- **#7** — TMR real-channel trials: `n=20 -> n=100` (tighter Wilson CI on the 0%-correction finding)
- **#8** — A second FABRIC site pair with different latency/congestion, to test whether "real network doesn't matter for silent divergence" holds beyond one low-noise link
- **#9** — 5 more real key pairs (5 -> 10), to de-risk Table 2 (finite-key length) from being sensitive to any one key

Each section is self-contained — run whichever you need. Sections B and C provision/reuse a
live FABRIC slice and will take a while (network round-trips); start them and let them run.

**Prerequisite:** existing repo checked out with `results/` populated from your original runs
(this notebook reads several of your existing CSVs to append to / compare against them).

## Section A — TMR: extend n=20 → n=100 (Feedback #7)

Reuses the exact `tmr_real_channel_trial` mechanism from
`notebooks/mitigation_v2_redundant_pa_tmr.ipynb`, at the same `fault_prob=0.3`, same key
(key0), just more trials. Appends to your existing `results/mitigation_tmr_realchannel.csv`
rather than overwriting it, so the first 20 trials you already collected are preserved.

In [ ]:
import sys, json, re, time
from pathlib import Path
import numpy as np
import pandas as pd
from galois import GF2
from randextract import ToeplitzHashing

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

import deploy_fabric as deploy
from qne.cascade import Key, ORIGINAL, Reconciliation, MockClassicalSession
from qne.cascade.key import key_from_sifted_json

RESULTS = PROJECT_DIR / "results"
print("Imports complete")


In [ ]:
# --- Connect to your existing slice and re-sync key0 (same as mitigation_v2 notebook) ---
SLICE_NAME = 'qfabric-bb84-2'

fablib = deploy.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show()

alice = slice_obj.get_node("alice")
bob = slice_obj.get_node("bob")
bob_ip = "10.10.1.2"

GEN_ALICE = RESULTS / "fabric_alice_sifted_bits_genonly.json"
GEN_BOB = RESULTS / "fabric_bob_sifted_bits_genonly.json"
META_PATH = RESULTS / "fabric_key0_pe_meta.json"

alice_key, alice_indices = key_from_sifted_json(str(GEN_ALICE), "alice_bits")
bob_key, bob_indices = key_from_sifted_json(str(GEN_BOB), "bob_bits")
assert alice_indices == bob_indices

meta = json.loads(META_PATH.read_text())
k_pe, real_qber = meta["k"], meta["qber"]
print(f"key0 loaded: n={alice_key.get_nr_bits()} bits, k={k_pe}, qber={real_qber:.4f}")

alice.upload_file(str(GEN_ALICE), "qfabric/results/alice_sifted_bits.json")
bob.upload_file(str(GEN_ALICE), "qfabric/results/alice_sifted_bits.json")
bob.upload_file(str(GEN_BOB), "qfabric/results/bob_sifted_bits.json")
print("Remote nodes re-synced to key0")


In [ ]:
def extract_output_path(stdout_text, marker):
    match = re.search(rf"{marker} (\S+\.json)", stdout_text)
    return match.group(1) if match else None


def tmr_real_channel_trial(bob, alice, bob_ip, real_qber, run, base_seed, fault_prob, k,
                            n_replicas=3, timeout=180):
    """Unchanged from mitigation_v2_redundant_pa_tmr.ipynb -- copied here so this
    notebook doesn't depend on that one having been run first in the same kernel."""
    replica_results = []
    for rep in range(n_replicas):
        seed = base_seed + rep * 1000
        bob_output = f"results/bob_tmr_run{run}_rep{rep}.json"

        bob_thread = bob.execute_thread(
            f"cd ~/qfabric && ~/qfabric/.venv/bin/python3 scripts/bob_cascade_driver.py "
            f"--key-json results/bob_sifted_bits.json --alice-key-json results/alice_sifted_bits.json "
            f"--host {bob_ip} --port 5200 --qber {real_qber} --k {k} --seed {seed} "
            f"--reconciliation-prob {fault_prob} --output {bob_output}"
        )
        alice_thread = alice.execute_thread(
            f"cd ~/qfabric && ~/qfabric/.venv/bin/python3 scripts/alice_cascade_responder.py "
            f"--key-json results/alice_sifted_bits.json --bob-host {bob_ip} --port 5200 "
            f"--seed {seed} --output results/alice_tmr_run{run}_rep{rep}.json"
        )

        bob_out = bob_thread.result(timeout=timeout)
        try:
            alice_thread.result(timeout=timeout // 2)
        except Exception:
            pass

        bob_output_path = extract_output_path(bob_out[0], "Result written to")
        if bob_output_path is None:
            replica_results.append({"status": "crashed", "bits": None})
            continue

        stdout, _ = bob.execute(f"cat ~/qfabric/{bob_output_path}", quiet=True)
        try:
            data = json.loads(stdout)
        except json.JSONDecodeError:
            replica_results.append({"status": "no_output", "bits": None})
            continue

        if data.get("non_convergent"):
            replica_results.append({"status": "non_convergent", "bits": None})
        elif "bob_reconciled_bits" in data:
            replica_results.append({
                "status": "converged", "bits": np.array(data["bob_reconciled_bits"]),
                "remaining_errors": data.get("remaining_errors_after_reconciliation"),
            })
        else:
            replica_results.append({"status": "converged_no_bits_saved", "bits": None})

        time.sleep(2)
    return replica_results

print("tmr_real_channel_trial defined")


In [ ]:
# --- Run 80 more trials (indices 20..99), continuing seeds from where the
# original 20 left off, and append to the SAME csv the paper's Table/Fig 9 was built from ---
TMR_CSV = RESULTS / "mitigation_tmr_realchannel.csv"
existing_tmr = pd.read_csv(str(TMR_CSV))
start_run = int(existing_tmr["run"].max()) + 1  # continues at 20 if you have the original 20
N_NEW_TRIALS = 80  # -> 100 total; raise/lower as time allows, CI tightens with more

new_rows = []
for run in range(start_run, start_run + N_NEW_TRIALS):
    base_seed = 42 + run * 10
    replicas = tmr_real_channel_trial(bob, alice, bob_ip, real_qber, run, base_seed,
                                       fault_prob=0.3, k=k_pe)
    statuses = [r["status"] for r in replicas]
    valid = [r["bits"] for r in replicas if r["bits"] is not None]

    if len(valid) < 2:
        new_rows.append({"run": run, "statuses": statuses, "result": "insufficient_valid_replicas",
                          "n_valid": len(valid), "corrected": False})
        print(f"run {run}: insufficient valid replicas ({statuses})")
        continue

    majority = (np.stack(valid).sum(axis=0) > len(valid) / 2).astype(np.uint8)
    n_errors = alice_key.nr_bits_different(Key(bits=majority))
    corrected = n_errors == 0
    new_rows.append({"run": run, "statuses": statuses, "result": "voted",
                      "n_valid": len(valid), "n_errors_after_tmr": n_errors, "corrected": corrected})
    print(f"run {run}: n_valid={len(valid)}, errors_after_vote={n_errors}, corrected={corrected}")

    # Checkpoint after every trial -- real-channel runs are slow and this can be
    # interrupted/resumed safely since start_run is read from the file each time.
    df_checkpoint = pd.concat([existing_tmr, pd.DataFrame(new_rows)], ignore_index=True)
    df_checkpoint.to_csv(str(TMR_CSV), index=False)

print(f"\nDone. Total trials in {TMR_CSV.name}: {len(df_checkpoint)}")


In [ ]:
# --- Recompute the Wilson 95% CI on the correction rate at n=100 (replaces the
# n=20, [0%, 16.1%] CI reported in the submitted draft) ---
from scipy.stats import norm

def wilson_ci(successes, n, z=1.96):
    if n == 0:
        return np.nan, np.nan, np.nan
    p_hat = successes / n
    denom = 1 + z**2 / n
    center = (p_hat + z**2 / (2 * n)) / denom
    halfwidth = (z / denom) * np.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2))
    return p_hat, max(0, center - halfwidth), min(1, center + halfwidth)

df_final = pd.read_csv(str(TMR_CSV))
voted = df_final[df_final["result"] == "voted"]
n_trials = len(voted)
n_corrected = int(voted["corrected"].sum())
p_hat, lo, hi = wilson_ci(n_corrected, n_trials)

mean_errors = voted["n_errors_after_tmr"].mean()
se_errors = voted["n_errors_after_tmr"].std() / np.sqrt(n_trials)

print(f"n={n_trials} real-channel TMR trials")
print(f"corrected: {n_corrected}/{n_trials}  (p_hat={p_hat:.3f})")
print(f"95% Wilson CI on true correction rate: [{lo*100:.1f}%, {hi*100:.1f}%]")
print(f"mean residual errors after vote: {mean_errors:.2f} +/- {se_errors:.2f} (SE)")


## Section B — Second FABRIC site pair (Feedback #8)

Your original pair was **NCSA <-> EDUKY** (Illinois <-> Kentucky): geographically close,
deliberately picked as "clear, robust, minimal noise." To test whether the mock-vs-real
non-effect (Section 4.3 / CMH p=0.16, p=0.32) generalizes, or was itself a property of that
particular low-noise link, we want a pair with **more distance and more hops**.

FABRIC's site list (per the project's public site info) includes coast-adjacent sites such
as `UTAH`, `MASS` (MGHPCC), `FIU`, `UCSD`, `MICH`, and `CLEM`, alongside the ESnet backbone
production sites (`STAR`/Starlight, `SALT`, `DALL`). Check live capacity before committing —
site availability on FABRIC fluctuates — with `fablib.list_sites()`.

**Suggested pair:** keep Alice + the BMv2 switch at `NCSA` (so Section A's key/switch config
carries over) and move Bob to `UTAH` — a genuinely different coast, more router hops, and a
different jitter profile than the NCSA-EDUKY link. Swap `SITE_BOB2` for whatever
`list_sites()` shows has capacity if `UTAH` doesn't.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

import deploy_fabric as deploy
from qne.config import ScenarioConfig
from qne.cascade.sweep_utils import run_real_channel_trial, collect_key_pairs

RESULTS = PROJECT_DIR / "results"
fablib = deploy.get_fablib()

# --- Sanity-check capacity before provisioning; adjust SITE_BOB2 based on what's free ---
for site in ["UTAH", "MASS", "FIU", "UCSD", "MICH", "CLEM", "STAR", "SALT"]:
    try:
        print(site, fablib.get_site_attribute(site, "cores_available"))
    except Exception as e:
        print(site, "unavailable:", e)


In [ ]:
SLICE_NAME_2 = 'qfabric-bb84-site2'
SITE_ALICE_2 = 'NCSA'
SITE_BOB_2   = 'UTAH'   # <- change based on the capacity check above if needed
SITE_SW_2    = 'NCSA'
SCENARIO     = 'validation/scenarios/fabric_1km.yml'
BMV2_IMAGE   = 'ghcr.io/kthare10/qfabric-bmv2:latest'

try:
    slice_obj_2 = fablib.get_slice(name=SLICE_NAME_2)
    print(f"Reusing existing slice '{SLICE_NAME_2}' (state: {slice_obj_2.get_state()})")
except Exception:
    slice_obj_2 = deploy.create_slice(fablib, SLICE_NAME_2, SITE_ALICE_2, SITE_BOB_2, SITE_SW_2)

deploy.upload_project(slice_obj_2)
import os
os.environ['QFABRIC_BMV2_IMAGE'] = BMV2_IMAGE
deploy.install_deps(slice_obj_2, build_bmv2=False)
deploy.setup_switch_docker(slice_obj_2, BMV2_IMAGE)

threshold = ScenarioConfig.from_yaml(PROJECT_DIR / SCENARIO).loss_threshold_u32
alice_mac_2, bob_mac_2, sw_alice_mac_2, sw_bob_mac_2, _, _ = deploy.configure_switch(slice_obj_2, threshold)
alice_ip_2, bob_ip_2 = deploy.setup_dataplane_ips(slice_obj_2, alice_mac_2, bob_mac_2)

alice_2 = slice_obj_2.get_node("alice")
bob_2 = slice_obj_2.get_node("bob")
print(f"\nSite-2 slice ready: {SITE_ALICE_2} <-> {SITE_BOB_2}, switch at {SITE_SW_2}")


In [ ]:
# --- Collect key pairs on the new link (5, matching your original 5-key design) ---
key_pairs_df_2 = collect_key_pairs(deploy, slice_obj_2, alice_2, bob_2, bob_ip_2, PROJECT_DIR,
                                    n_keys=5, scenario_path=SCENARIO)
# NOTE: collect_key_pairs writes results/key_pairs_metadata.csv, which would clobber
# Section C's site-1 metadata file if run in the same results/ dir. Rename immediately:
import shutil
shutil.move(str(RESULTS / "key_pairs_metadata.csv"), str(RESULTS / "key_pairs_metadata_site2.csv"))
for i in range(5):
    for who in ["alice", "bob"]:
        for suffix in ["", "_raw"]:
            src = RESULTS / f"{who}_sifted_bits_key{i}{suffix}.json"
            if src.exists():
                shutil.move(str(src), str(RESULTS / f"{who}_sifted_bits_key{i}{suffix}_site2.json"))
print(key_pairs_df_2)


In [ ]:
# --- Same dose-response sweep as your paper's real-channel data (Section 3.5 /
# results/sdc_real_channel_toeplitz_finalkey_doseresponse_multikey_asymptotic.csv),
# run on the site-2 link instead. Lower N_RUNS_PER_PROB from the paper's n=150/n=500
# to start -- each point is a live FABRIC round trip; raise it once you've confirmed
# the pipeline runs cleanly end-to-end.
#
# NOTE: we upload each key under its OWN remote filename (alice/bob_sifted_bits_key{i}.json)
# and pass those paths straight to run_real_channel_trial via bob_key_json/alice_key_json,
# instead of re-uploading into the generic "results/alice_sifted_bits.json" name every
# loop iteration. That generic file is also written natively on the node by
# deploy.run_bb84() during collect_key_pairs_from above, and can end up owned/permissioned
# such that a later SFTP overwrite from this session gets a PermissionError (errno 13) --
# per-key filenames sidestep the clash entirely, and match how collect_key_pairs already
# names files on the remote side. ---
PROBS = [0.5, 0.3, 0.1, 0.05, 0.03, 0.01, 0.003, 0.001]
N_RUNS_PER_PROB = 20  # paper used 150 for toeplitz/final-key; raise as time allows
CSV_SITE2 = RESULTS / "sdc_site2_toeplitz_finalkey_doseresponse.csv"

all_rows = []
for key_idx in range(5):
    meta_row = key_pairs_df_2.iloc[key_idx]
    k_i, qber_i = int(meta_row["k_pe"]), float(meta_row["qber"])

    a_file = RESULTS / f"alice_sifted_bits_key{key_idx}_site2.json"
    b_file = RESULTS / f"bob_sifted_bits_key{key_idx}_site2.json"

    a_remote = f"results/alice_sifted_bits_key{key_idx}_site2.json"
    b_remote = f"results/bob_sifted_bits_key{key_idx}_site2.json"
    alice_2.upload_file(str(a_file), f"qfabric/{a_remote}")
    bob_2.upload_file(str(a_file), f"qfabric/{a_remote}")
    bob_2.upload_file(str(b_file), f"qfabric/{b_remote}")
    print(f"\n=== site2 key{key_idx}: qber={qber_i:.4f} ===")

    for prob in PROBS:
        for fault_type, kwargs in [("toeplitz", {"toeplitz_prob": prob}),
                                    ("final_key", {"final_key_prob": prob})]:
            for run in range(N_RUNS_PER_PROB):
                seed = 42 + run
                result = run_real_channel_trial(bob_2, alice_2, bob_ip_2, qber_i, run, seed, k=k_i,
                                                   length_mode="asymptotic",
                                                   bob_key_json=b_remote, alice_key_json=a_remote,
                                                   **kwargs)
                result["fault_type"] = fault_type
                result["prob"] = prob
                result["key_index"] = key_idx
                result["qber"] = qber_i
                all_rows.append(result)
            # checkpoint per (key, prob, fault_type) block
            pd.DataFrame(all_rows).to_csv(str(CSV_SITE2), index=False)

print(f"\nSaved {len(all_rows)} rows -> {CSV_SITE2}")


In [ ]:
# --- Site-1 vs. site-2 CMH test, matched to your existing mock-vs-real methodology
# (same cmh_manual / stratify-by-prob pattern as notebooks/sdc_sweeps_all5keys.ipynb) ---
from scipy.stats import chi2

def cmh_manual(tables):
    num, den = 0.0, 0.0
    for t in tables:
        a, b, c, d = t[0, 0], t[0, 1], t[1, 0], t[1, 1]
        n = a + b + c + d
        if n < 2:
            continue
        row1, row2, col1, col2 = a + b, c + d, a + c, b + d
        E_a = row1 * col1 / n
        var_a = row1 * row2 * col1 * col2 / (n**2 * (n - 1))
        num += (a - E_a)
        den += var_a
    if den == 0:
        return np.nan, np.nan
    stat = num**2 / den
    return stat, 1 - chi2.cdf(stat, 1)


def build_tables_silent_both(site1_df, site2_df, fault_type):
    """One 2x2 table per firing-probability stratum, restricting BOTH sides
    to silent (verification-passed) mismatches, same as your mock-vs-real test."""
    tables = []
    for p in sorted(site2_df["prob"].unique()):
        s1 = site1_df[(site1_df["prob"] == p)]
        s2 = site2_df[(site2_df["prob"] == p) & (site2_df["fault_type"] == fault_type)]
        if len(s1) == 0 or len(s2) == 0:
            continue
        s1_mis = ((~s1["keys_match"].astype(bool)) & (s1["verification_passed"] == True)).sum()
        s2_mis = ((~s2["keys_match"].astype(bool)) & (s2["verification_passed"] == True)).sum()
        tables.append(np.array([[s1_mis, len(s1) - s1_mis], [s2_mis, len(s2) - s2_mis]]))
    return tables

site2 = pd.read_csv(str(CSV_SITE2))

# Your existing site-1 (NCSA-EDUKY) real-channel CSVs from the original submission:
site1_toeplitz = pd.read_csv(str(RESULTS / "sdc_realchannel_toeplitz_allkeys_asymptotic.csv"))
site1_finalkey = pd.read_csv(str(RESULTS / "sdc_realchannel_finalkey_allkeys_asymptotic.csv"))

print("=== Site 1 (NCSA<->EDUKY) vs. Site 2 (NCSA<->UTAH), pooled CMH test ===\n")
for name, site1_df, ftype in [("toeplitz", site1_toeplitz, "toeplitz"),
                               ("final_key", site1_finalkey, "final_key")]:
    tables = build_tables_silent_both(site1_df, site2, ftype)
    stat, p = cmh_manual(tables)
    n1 = sum(t[0].sum() for t in tables)
    n2 = sum(t[1].sum() for t in tables)
    print(f"{name}: n_site1={n1}, n_site2={n2}, CMH stat={stat:.3f}, p={p:.4f} "
          f"-- {'SIGNIFICANT (site matters)' if p < 0.05 else 'not significant (consistent with Section 4.3)'}")


## Section C — 5 more real keys, 5 → 10 (Feedback #9)

Extends `collect_key_pairs` with a `start_index`, so the new keys land at indices 5-9
without touching your existing `key_pairs_metadata.csv` (indices 0-4) or the `key{i}` JSON
files that Sections 4-8 of the paper already depend on. Then recomputes Table 2
(finite-key vs. asymptotic length) for all 10 keys, so `ell_finite`'s sensitivity to any
single key (key 1 at `ell=1` had to be excluded as degenerate) is easier to characterize.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

import deploy_fabric as deploy
from qne.cascade.key import key_from_sifted_json
from qne.cascade.parameter_estimation import split_pe_and_generation
from qne.cascade.finite_key import finite_key_output_length, asymptotic_key_length

RESULTS = PROJECT_DIR / "results"

SLICE_NAME = 'qfabric-bb84-2'  # site-1 slice, same as Section A / the original 5 keys
fablib = deploy.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
alice = slice_obj.get_node("alice")
bob = slice_obj.get_node("bob")
bob_ip = "10.10.1.2"


In [ ]:
def collect_key_pairs_from(deploy, slice_obj, alice, bob, bob_ip, project_dir,
                           start_index, n_keys, scenario_path="validation/scenarios/fabric_1km.yml",
                           pe_sample_fraction=0.1, pe_seed_base=100000):
    """Same body as qne.cascade.sweep_utils.collect_key_pairs, but starts at
    start_index and APPENDS to key_pairs_metadata.csv instead of overwriting it --
    so your existing 5 keys (indices 0-4) are preserved."""
    alice_mac = alice.get_interface(network_name="net_alice_switch").get_mac()
    bob_mac = bob.get_interface(network_name="net_switch_bob").get_mac()
    sw_alice_mac = slice_obj.get_node("switch").get_interface(network_name="net_alice_switch").get_mac()

    key_pairs = []
    for i in range(start_index, start_index + n_keys):
        print(f"\n=== Collecting key pair {i} ===")
        deploy.run_bb84(slice_obj, scenario_path, alice_mac, bob_mac,
                          sw_alice_mac=sw_alice_mac, bob_data_ip=bob_ip)

        stdout, _ = alice.execute("cat ~/qfabric/results/alice_sifted_bits.json", quiet=True)
        alice_raw_path = project_dir / "results" / f"alice_sifted_bits_key{i}_raw.json"
        alice_raw_path.write_text(stdout)

        stdout, _ = bob.execute("cat ~/qfabric/results/bob_sifted_bits.json", quiet=True)
        bob_raw_path = project_dir / "results" / f"bob_sifted_bits_key{i}_raw.json"
        bob_raw_path.write_text(stdout)

        alice_key_full, alice_idx_full = key_from_sifted_json(str(alice_raw_path), "alice_bits")
        bob_key_full, bob_idx_full = key_from_sifted_json(str(bob_raw_path), "bob_bits")

        if alice_idx_full != bob_idx_full:
            print(f"  key pair {i}: index mismatch -- skipping (raw files saved for inspection)")
            continue

        try:
            split = split_pe_and_generation(alice_key_full, bob_key_full,
                                             sample_fraction=pe_sample_fraction, seed=pe_seed_base + i)
        except ValueError as e:
            print(f"  key pair {i}: PE split failed ({e}) -- skipping")
            continue

        m, k, n = split["m"], split["k"], split["n"]
        qber_i = split["qber"]
        alice_key_i, bob_key_i = split["alice_gen"], split["bob_gen"]
        print(f"  key pair {i}: m={m} sifted, k={k} PE sample, n={n} generation bits, qber={qber_i:.4f}")

        alice_path = project_dir / "results" / f"alice_sifted_bits_key{i}.json"
        bob_path = project_dir / "results" / f"bob_sifted_bits_key{i}.json"
        gen_indices = split["gen_indices"]
        alice_matching_indices_gen = [alice_idx_full[idx] for idx in gen_indices]

        alice_path.write_text(json.dumps({"alice_bits": alice_key_i.bits.tolist(),
                                           "matching_indices": alice_matching_indices_gen}))
        bob_path.write_text(json.dumps({"bob_bits": bob_key_i.bits.tolist(),
                                         "matching_indices": alice_matching_indices_gen}))

        key_pairs.append({"index": i, "m_sifted": m, "k_pe": k, "n_bits": n, "qber": qber_i,
                           "alice_path": str(alice_path), "bob_path": str(bob_path),
                           "alice_raw_path": str(alice_raw_path), "bob_raw_path": str(bob_raw_path)})

        bob.upload_file(str(alice_path), f"qfabric/results/alice_sifted_bits_key{i}.json")
        bob.upload_file(str(bob_path), f"qfabric/results/bob_sifted_bits_key{i}.json")

    return pd.DataFrame(key_pairs)


existing_meta = pd.read_csv(str(RESULTS / "key_pairs_metadata.csv"))
new_keys_df = collect_key_pairs_from(deploy, slice_obj, alice, bob, bob_ip, PROJECT_DIR,
                                      start_index=5, n_keys=5)
combined_meta = pd.concat([existing_meta, new_keys_df], ignore_index=True)
combined_meta.to_csv(str(RESULTS / "key_pairs_metadata.csv"), index=False)
print(f"\nkey_pairs_metadata.csv now has {len(combined_meta)} keys (indices "
      f"{combined_meta['index'].min()}-{combined_meta['index'].max()})")


In [ ]:
# --- Recompute Table 2 (finite-key vs. asymptotic length) for all 10 keys ---
rows = []
for _, row in combined_meta.iterrows():
    i, n, k, qber = int(row["index"]), int(row["n_bits"]), int(row["k_pe"]), float(row["qber"])
    ell_asymptotic = asymptotic_key_length(n, qber)
    try:
        ell_finite, r, t, nu = finite_key_output_length(n, k, qber)
    except ValueError as e:
        ell_finite = None
        print(f"key {i}: finite-key optimization failed ({e})")
    reduction = (1 - ell_finite / ell_asymptotic) * 100 if ell_finite and ell_asymptotic else None
    rows.append({"key": i, "n_bits": n, "qber": qber,
                 "ell_asymptotic": ell_asymptotic, "ell_finite": ell_finite, "reduction_pct": reduction})

table2_extended = pd.DataFrame(rows)
table2_extended.to_csv(str(RESULTS / "finitekey_vs_asymptotic_table2_10keys.csv"), index=False)
print(table2_extended.to_string(index=False))
print(f"\nOriginal 5-key mean reduction: {table2_extended.iloc[:5]['reduction_pct'].mean():.2f}%")
print(f"All-10-key mean reduction:     {table2_extended['reduction_pct'].mean():.2f}%")
print("(Exclude any key with an extreme ell_finite, e.g. <=1 bit, from the mean the same "
      "way key 1 was excluded in the original 5-key table.)")
